# Modeling: 배터리 수명 예측 모델 개발 및 평가

설계된 Feature Sets를 바탕으로 최소 기준 모델(Ridge 회귀 파이프라인) 구축, 학습, 평가 및 공식 리포트 생성

## 0. 필수 라이브러리 및 경로 설정

In [1]:
import json
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.model_selection import GroupKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings('ignore')
plt.rcParams['font.family'] = 'AppleGothic'
plt.rcParams['axes.unicode_minus'] = False

# 경로 설정
CURRENT_DIR = Path.cwd().resolve()
ROOT = CURRENT_DIR.parent if CURRENT_DIR.name in ('notebooks', 'test') else CURRENT_DIR
DATA_DIR = ROOT / 'results' / 'features'
OUTPUT_DIR = ROOT / 'results'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f'데이터 경로: {DATA_DIR}')
print(f'산출물 저장 경로: {OUTPUT_DIR}')

데이터 경로: /Users/yujin.je/Workspace/ds-mini/results/features
산출물 저장 경로: /Users/yujin.je/Workspace/ds-mini/results


## 1. 데이터 로드 및 평가 지표 정의
- **데이터 로드**: 02_feature_engineering에서 분할 저장된 Raw 데이터셋 및 `feature_sets.json` 로드
- **주요 평가 지표**: *Nature Energy* 원논문 벤치마크 기준인 MAPE(목표 9.1%) 산출 함수 정의

In [2]:
# 데이터셋 로드
train_df = pd.read_csv(DATA_DIR / 'train_features_raw.csv')
val_df = pd.read_csv(DATA_DIR / 'valid_features_raw.csv')
test_b2 = pd.read_csv(DATA_DIR / 'test_b2_features_raw.csv')
test_b3 = pd.read_csv(DATA_DIR / 'test_b3_features_raw.csv')

with open(DATA_DIR / 'feature_sets.json', 'r', encoding='utf-8') as f:
    FEATURE_SETS = json.load(f)

# 타겟 정의
y_train = train_df['cycle_life'].values
y_val = val_df['cycle_life'].values
y_b2 = test_b2['cycle_life'].values
y_b3 = test_b3['cycle_life'].values

# MAPE 산출 함수
def calculate_mape(y_true, y_pred):
    return np.mean(np.abs((y_true - y_pred) / y_true)) * 100

print(f'Train: {len(train_df)}개 | Valid: {len(val_df)}개 | Test B2: {len(test_b2)}개 | Test B3: {len(test_b3)}개')
print(f'Feature Sets: {list(FEATURE_SETS.keys())}')

Train: 35개 | Valid: 11개 | Test B2: 39개 | Test B3: 44개
Feature Sets: ['A', 'B', 'C', 'D', 'ALT_delta_q_min', 'E_mean_QD']


## 2. 모델 학습 및 교차 검증
- **파이프라인 구축**: 결측 대치(SimpleImputer), 표준화(StandardScaler), Ridge 회귀 결합
- **누수 차단 검증**: Train 세트 내부 `charging_policy` 기준 `GroupKFold(n_splits=4)` 교차 검증 및 Hold-out Valid/외부 Test 평가

In [3]:
selected_features = FEATURE_SETS['A']

# Train CV 평가 함수
def evaluate_pipeline_cv(model, X, y, groups, n_splits=4):
    gkf = GroupKFold(n_splits=n_splits)
    scores = []
    for tr_idx, val_idx in gkf.split(X, y, groups=groups):
        pipe = Pipeline([
            ('imputer', SimpleImputer(strategy='median')),
            ('scaler', StandardScaler()),
            ('model', model)
        ])
        pipe.fit(X.iloc[tr_idx], y[tr_idx])
        pred = pipe.predict(X.iloc[val_idx])
        scores.append(calculate_mape(y[val_idx], pred))
    return np.mean(scores)

# 최종 모델 파이프라인
ridge_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler()),
    ('model', Ridge(alpha=1.0, random_state=42))
])

# 학습
ridge_pipeline.fit(train_df[selected_features], y_train)

# 예측
train_cv_mape = evaluate_pipeline_cv(Ridge(alpha=1.0, random_state=42), train_df[selected_features], y_train, train_df['charging_policy'])
valid_pred = ridge_pipeline.predict(val_df[selected_features])
b2_pred = ridge_pipeline.predict(test_b2[selected_features])
b3_pred = ridge_pipeline.predict(test_b3[selected_features])

valid_mape = calculate_mape(y_val, valid_pred)
test_b2_mape = calculate_mape(y_b2, b2_pred)
test_b3_mape = calculate_mape(y_b3, b3_pred)

print(f'Train CV MAPE: {train_cv_mape:.2f}%')
print(f'Valid MAPE: {valid_mape:.2f}%')
print(f'Test B2 MAPE: {test_b2_mape:.2f}%')
print(f'Test B3 MAPE: {test_b3_mape:.2f}%')

Train CV MAPE: 7.08%
Valid MAPE: 12.72%
Test B2 MAPE: 25.85%
Test B3 MAPE: 12.16%


## 3. 모델 성능 리포팅
- `pjt-info.md` 표준 양식에 맞추어 Train, Valid, Test B2, Test B3 성능 및 Gap 테이블 생성 및 CSV 저장

In [4]:
TARGET_MAPE = 9.1

report_rows = [
    {'구분': 'Train (Batch 1 CV)', 'MAPE (%)': f'{train_cv_mape:.2f}', '비고': ''},
    {'구분': 'Valid (Batch 1 Hold-out)', 'MAPE (%)': f'{valid_mape:.2f}', '비고': ''},
    {'구분': 'Test (Batch 2)', 'MAPE (%)': f'{test_b2_mape:.2f}', '비고': ''},
    {'구분': '  Gap (Train-Valid)', 'MAPE (%)': f'{valid_mape - train_cv_mape:+.2f}', '비고': '(+) : 과적합 의심'},
    {'구분': '  Gap (Valid-Test)', 'MAPE (%)': f'{test_b2_mape - valid_mape:+.2f}', '비고': '(+) : 배치간 일반화 저하 의심'},
    {'구분': '  Gap (Target-Test)', 'MAPE (%)': f'{test_b2_mape - TARGET_MAPE:+.2f}', '비고': 'Target : 원논문 9.1%'},
    {'구분': 'Test (Batch 3)', 'MAPE (%)': f'{test_b3_mape:.2f}', '비고': ''},
    {'구분': '  Gap (Batch2-Batch3)', 'MAPE (%)': f'{test_b3_mape - test_b2_mape:+.2f}', '비고': '(+) : Batch 3 성능 저하'},
    {'구분': '  Gap (Target-Test)', 'MAPE (%)': f'{test_b3_mape - TARGET_MAPE:+.2f}', '비고': 'Batch 3 기준, 원논문 9.1% 비교'},
]

report_df = pd.DataFrame(report_rows)
display(report_df)

report_df.to_csv(OUTPUT_DIR / 'model_performance.csv', index=False, encoding='utf-8-sig')
print(f'성능 리포트 저장 완료: {OUTPUT_DIR / "model_performance.csv"}')

,구분,MAPE (%),비고
0,Train (Batch 1 CV),7.08,
1,Valid (Batch 1 Hold-out),12.72,
2,Test (Batch 2),25.85,
3,Gap (Train-Valid),+5.64,(+) : 과적합 의심
4,Gap (Valid-Test),+13.12,(+) : 배치간 일반화 저하 의심
5,Gap (Target-Test),+16.75,Target : 원논문 9.1%
6,Test (Batch 3),12.16,
7,Gap (Batch2-Batch3),-13.69,(+) : Batch 3 성능 저하
8,Gap (Target-Test),+3.06,"Batch 3 기준, 원논문 9.1% 비교"


성능 리포트 저장 완료: /Users/yujin.je/Workspace/ds-mini/results/model_performance.csv
